# OfficeNet — Mulai Cepat / Quick Start

**ID** — Word, Excel, PowerPoint, dan PDF untuk .NET 10. Notebook ini menyentuh keempatnya dalam
beberapa sel. Untuk pendalaman, buka notebook per library.

**EN** — Word, Excel, PowerPoint and PDF for .NET 10. This notebook touches all four in a few
cells. For depth, open the per-library notebooks.

> Dibuat oleh Gravicode Studios, dipimpin oleh Kang Fadhil.

Jalankan `dotnet build OfficeNet.sln -c Release` sekali sebelum sel pertama. /
Run `dotnet build OfficeNet.sln -c Release` once before the first cell.

In [ ]:
// Build first:  dotnet build OfficeNet.sln -c Release

#r "../src/OfficeNet.Core/bin/Release/net10.0/Gravicode.OfficeNet.Core.dll"
#r "../src/PdfNet/bin/Release/net10.0/Gravicode.OfficeNet.PdfNet.dll"
#r "../src/WordNet/bin/Release/net10.0/Gravicode.OfficeNet.WordNet.dll"
#r "../src/ExcelNet/bin/Release/net10.0/Gravicode.OfficeNet.ExcelNet.dll"
#r "../src/PowerPointNet/bin/Release/net10.0/Gravicode.OfficeNet.PowerPointNet.dll"
#r "../src/OfficeNet/bin/Release/net10.0/Gravicode.OfficeNet.dll"
#r "../src/OfficeNet.Rendering/bin/Release/net10.0/Gravicode.OfficeNet.Rendering.dll"

// Published instead? Swap the lines above for:
//   #r "nuget: Gravicode.OfficeNet, *"

In [ ]:
using OfficeNet.Rendering;
using Microsoft.DotNet.Interactive.Formatting;

// Renders a document and shows the first page inline, so a cell's effect is visible rather than
// described. Base64 in an <img> because the notebook has nowhere to serve a file from.
void Show(string path, int width = 520)
{
    var png = DocumentRenderer.RenderThumbnail(path, width);
    var data = Convert.ToBase64String(png);

    display(HTML($"<img src='data:image/png;base64,{data}' style='border:1px solid #ddd' />"));
}

var work = Path.Combine(Path.GetTempPath(), "officenet-notebook");
Directory.CreateDirectory(work);
string At(string name) => Path.Combine(work, name);

Console.WriteLine($"Berkas ditulis ke / files written to: {work}");

## Word

Satu dokumen dengan heading, paragraf, dan tabel. /
One document with a heading, a paragraph and a table.

In [ ]:
using WordNet;
using OfficeNet.Core;
using OfficeNet.Core.Drawing;

using (var document = WordDocument.Create())
{
    document.AddHeading("Laporan Tahunan", 0);
    document.AddParagraph("Dibuat oleh Gravicode Studios.", "Subtitle");

    document.AddHeading("Pendapatan", 1);

    var table = document.AddTable(new[]
    {
        new[] { "Wilayah", "2025", "2026" },
        new[] { "Jakarta", "1.120", "1.480" },
        new[] { "Bandung", "860", "1.150" },
    });

    table.Rows[0].SetShading(OfficeColor.FromRgb(0x1F, 0x38, 0x64));

    foreach (var cell in table.Rows[0].Cells)
        foreach (var run in cell.Paragraphs.SelectMany(p => p.Runs))
            run.Format.Color = OfficeColor.White;

    document.Save(At("laporan.docx"));
}

Show(At("laporan.docx"));

## Excel

Perhatikan `Recalculate()`: tanpa itu, hasil formula tersimpan sebagai nol untuk setiap konsumen
selain Excel. /
Note `Recalculate()`: without it the cached results are zero for every consumer except Excel.

In [ ]:
using ExcelNet;
using ExcelNet.Styles;

using (var workbook = Workbook.Create("Penjualan"))
{
    var sheet = workbook["Penjualan"];
    sheet.WriteHeader("A1", ["Produk", "Qty", "Harga", "Total"]);

    string[] products = ["WordNet", "ExcelNet", "PowerPointNet", "PdfNet"];

    for (var i = 0; i < products.Length; i++)
    {
        var row = i + 1;
        sheet[row, 0].Set(products[i]);
        sheet[row, 1].Set((i + 2) * 7);
        sheet[row, 2].Set(150_000.0).WithNumberFormat(NumberFormats.Rupiah);
        sheet[row, 3].SetFormula($"B{row + 1}*C{row + 1}").WithNumberFormat(NumberFormats.Rupiah);
    }

    sheet["C6"].Set("TOTAL").Bold();
    sheet["D6"].SetFormula("SUM(D2:D5)").WithNumberFormat(NumberFormats.Rupiah);

    sheet.AutoFitColumns();
    workbook.Recalculate();

    Console.WriteLine($"Total = {sheet["D6"].Number:N0}");
    workbook.Save(At("penjualan.xlsx"));
}

Show(At("penjualan.xlsx"), 640);

## PowerPoint

Chart-nya native: angkanya ikut serta di dalam berkas. /
The chart is native: the numbers travel inside the file.

In [ ]:
using PowerPointNet;
using PowerPointNet.Charts;
using OfficeNet.Core.Charts;

using (var deck = Presentation.Create())
{
    deck.AddTitleSlide("OfficeNet", "Empat format, satu API");

    // Layout 2 is "Title Only": the title sits at the top and the rest of the slide is free.
    var slide = deck.AddSlide(2);
    slide.SetTitle("Pendapatan per Wilayah");

    slide.AddChart(new ChartData
    {
        Type = ChartType.Column,
        Categories = ["Jakarta", "Bandung", "Surabaya"],
        Series = [new ChartSeries("2026", [1480, 1150, 905])],
        ValueFormat = "#,##0",
        ShowDataLabels = true,
    });

    deck.Save(At("deck.pptx"));
}

Show(At("deck.pptx"), 640);

## PDF, dan konversi lintas format / PDF, and cross-format conversion

In [ ]:
using OfficeNet;

foreach (var name in new[] { "laporan.docx", "penjualan.xlsx", "deck.pptx" })
{
    var pdf = Office.ConvertToPdf(At(name));
    Console.WriteLine($"{name,-20} -> {Path.GetFileName(pdf)}  ({new FileInfo(pdf).Length / 1024.0:0.0} KB)");
}

Console.WriteLine();
var text = Office.ExtractText(At("laporan.docx"));
Console.WriteLine(text[..Math.Min(120, text.Length)] + "…");

## Selanjutnya / Next

- [`WordNet.ipynb`](WordNet.ipynb) · [`ExcelNet.ipynb`](ExcelNet.ipynb) ·
  [`PowerPointNet.ipynb`](PowerPointNet.ipynb) · [`PdfNet.ipynb`](PdfNet.ipynb)
- Dokumentasi lengkap: [`docs/`](../docs/README.md) — [Bahasa Indonesia](../docs/id/README.md)